<a href="https://colab.research.google.com/github/noname123920/diplom/blob/main/PINN2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
def relative_l2_error(model, exact_fn, n_x=200, n_t=100,
                      domain_x=(0, np.pi), domain_t=(0, 1)):
    x = np.linspace(domain_x[0], domain_x[1], n_x)
    t = np.linspace(domain_t[0], domain_t[1], n_t)
    X, T = np.meshgrid(x, t)
    inputs = np.stack([X.ravel(), T.ravel()], axis=1).astype(np.float32) # расчетная сетка из 20000 точек (200x100)
    p_pinn  = model(inputs).numpy().reshape(X.shape)
    p_exact = exact_fn(X, T)
    return np.linalg.norm(p_pinn - p_exact) / np.linalg.norm(p_exact)

## 1.1. Piezoconductivity Equation (Dirichlet)

Solve the 1D piezoconductivity equation: $p_t = \kappa p_{xx}$ for $x$ in $[0, \pi]$, $t$ in $[0, 1]$, with initial condition $p(x, 0) = \sin(x)$, boundary conditions $p(0,t) = 0$, $p(\pi,t) = 0$.

In [2]:
import tensorflow as tf
import numpy as np

# Parameters
kappa = 1
domain_x = [0, np.pi]
domain_t = [0, 1]

# Neural Network
class PINN(tf.keras.Model):
  def __init__(self):
    super(PINN, self).__init__()
    self.dense1 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense2 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense3 = tf.keras.layers.Dense(1)

  def call(self, inputs):
    x = inputs[:, 0:1]
    t = inputs[:, 1:2]

    u = self.dense1(inputs)
    u = self.dense2(u)
    u = self.dense3(u)

    return tf.sin(x) + x * (np.pi - x) * t * u

# PDE residual
def pde_residual(model, x, t):
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x)
    tape.watch(t)
    inputs = tf.concat([x, t], axis=1)
    p = model(inputs)
    p_t = tape.gradient(p, t)
    p_x = tape.gradient(p, x)
  p_xx = tape.gradient(p_x, x)
  del tape
  return p_t - kappa * p_xx

# Training
model = PINN()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
loss_history_dirichlet = []
for epoch in range(1000):
  x = tf.random.uniform((100, 1), domain_x[0], domain_x[1])
  t = tf.random.uniform((100, 1), domain_t[0], domain_t[1])
  with tf.GradientTape() as tape:
    residual = pde_residual(model, x, t)
    loss_pde = tf.reduce_mean(tf.square(residual))
    loss_dirichlet = loss_pde
  grads = tape.gradient(loss_dirichlet, model.trainable_variables)
  optimizer.apply_gradients(zip(grads, model.trainable_variables))
  if epoch % 50 == 0:
    loss_val = float(loss_dirichlet.numpy())
    loss_history_dirichlet.append(loss_val)
    print(f"Epoch {epoch}, Loss: {loss_dirichlet.numpy()}")

err_dirichlet = relative_l2_error(model, lambda x, t: np.exp(-t) * np.sin(x))
print(f"Dirichlet: relative L2 error = {err_dirichlet:.4e}")

Epoch 0, Loss: 8.26049518585205
Epoch 50, Loss: 0.22418315708637238
Epoch 100, Loss: 0.10605771839618683
Epoch 150, Loss: 0.055078376084566116
Epoch 200, Loss: 0.021072296425700188
Epoch 250, Loss: 0.006989104673266411
Epoch 300, Loss: 0.0033236113376915455
Epoch 350, Loss: 0.003671559039503336
Epoch 400, Loss: 0.0027474365197122097
Epoch 450, Loss: 0.001870740670710802
Epoch 500, Loss: 0.0017534721409901977
Epoch 550, Loss: 0.001903332071378827
Epoch 600, Loss: 0.0015335833886638284
Epoch 650, Loss: 0.0016048343386501074
Epoch 700, Loss: 0.0013974786270409822
Epoch 750, Loss: 0.0011082337005063891
Epoch 800, Loss: 0.0012250464642420411
Epoch 850, Loss: 0.00094201898900792
Epoch 900, Loss: 0.0010160355595871806
Epoch 950, Loss: 0.0008627992938272655
Dirichlet: relative L2 error = 7.1936e-03


In [3]:
import plotly.express as px
import pandas as pd

time_steps = np.linspace(domain_t[0], domain_t[1], 51)
plot_data = []

for t_val in time_steps:
  x_plot = np.linspace(domain_x[0], domain_x[1], 100)[:, None]
  t_plot = t_val * np.ones_like(x_plot)
  inputs_plot = np.concatenate([x_plot, t_plot], axis=1)
  p_plot = model(inputs_plot).numpy()
  p_exact = np.exp(-t_val) * np.sin(x_plot)
  df_temp = pd.DataFrame({
      'x': x_plot.flatten(),
      'PINN': p_plot.flatten(),
      'Exact': p_exact.flatten(),
      'Time': t_val
  })
  plot_data.append(df_temp)

df_plot = pd.concat(plot_data)
df_melted = df_plot.melt(id_vars=['x', 'Time'], var_name='Solution Type', value_name='p')
fig = px.line(df_melted, x='x', y='p', animation_frame='Time', color='Solution Type',
              title='1D Piezoconductivity Equation Solution Over Time')
fig.update_layout(
    xaxis_title='x',
    yaxis_title='p(x, t)'
)
fig.show()

In [43]:
import numpy as np
import plotly.graph_objects as go

# Сетка (твои параметры)
n_x, n_t = 200, 100
x = np.linspace(domain_x[0], domain_x[1], n_x)
t = np.linspace(domain_t[0], domain_t[1], n_t)
X, T = np.meshgrid(x, t)

inputs = np.stack([X.ravel(), T.ravel()], axis=1).astype(np.float32)
P_pinn = model(inputs).numpy().reshape(X.shape)

fig = go.Figure(go.Surface(
    x=X, y=T, z=P_pinn,
    colorscale='RdBu_r',
    colorbar=dict(title='p(x, t)'),
    hovertemplate=(
        'x = %{x:.3f}<br>' +
        't = %{y:.3f}<br>' +
        'p = %{z:.4f}<extra></extra>'
    )
))

fig.update_layout(
    title='Давление p(x, t) — PINN',
    scene=dict(
        xaxis_title='x',
        yaxis_title='t',
        zaxis_title='p(x, t)',
        camera=dict(eye=dict(x=1.8, y=1.8, z=1.8))
    )
)
fig.show()

## LOSS:

In [4]:
import plotly.graph_objects as go

epochs = [0, 50, 100, 150, 200, 250, 300, 350, 400, 450, 500,
          550, 600, 650, 700, 750, 800, 850, 900, 950]

fig = go.Figure()
fig.add_trace(go.Scatter(x=epochs, y=loss_history_dirichlet, mode='lines+markers', name='Дирихле'))
fig.update_layout(
    title='Сходимость обучения PINN',
    xaxis_title='Эпоха',
    yaxis_title='Функция потерь',
    yaxis_type='log',  # логарифмическая шкала
    template='plotly_white'
)
fig.show()

## Что будет если менять число эпох?

In [11]:
import tensorflow as tf
import numpy as np

tf.random.set_seed(42)
np.random.seed(42)

checkpoints = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]
max_epochs = 10000

model = PINN()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)

errors = {}
losses = {}

for epoch in range(1, max_epochs + 1):
    x = tf.random.uniform((100, 1), domain_x[0], domain_x[1])
    t = tf.random.uniform((100, 1), domain_t[0], domain_t[1])
    with tf.GradientTape() as tape:
        loss = tf.reduce_mean(tf.square(pde_residual(model, x, t)))
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    if epoch in checkpoints:
        err = relative_l2_error(model, lambda x, t: np.exp(-t) * np.sin(x))
        errors[epoch] = err
        losses[epoch] = float(loss.numpy())
        print(f"Эпоха {epoch:>5} → L2 = {err:.4e}, loss = {float(loss):.3e}")

Эпоха  1000 → L2 = 1.2644e-02, loss = 1.416e-03
Эпоха  2000 → L2 = 6.7205e-03, loss = 5.353e-04
Эпоха  3000 → L2 = 3.7242e-03, loss = 2.054e-04
Эпоха  4000 → L2 = 2.9440e-03, loss = 9.466e-05
Эпоха  5000 → L2 = 2.4195e-03, loss = 8.833e-05
Эпоха  6000 → L2 = 1.1507e-03, loss = 5.633e-05
Эпоха  7000 → L2 = 4.7593e-03, loss = 3.800e-05
Эпоха  8000 → L2 = 2.8624e-03, loss = 2.894e-05
Эпоха  9000 → L2 = 1.3542e-03, loss = 1.834e-05
Эпоха 10000 → L2 = 2.2995e-03, loss = 1.289e-05


In [13]:
import tensorflow as tf
import numpy as np

checkpoints = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]
max_epochs = 10000
seeds = [1, 2, 3, 4, 5]

all_errors = {}

for seed in seeds:
    print(f"\n===== Прогон {seed} / 5 =====")
    tf.random.set_seed(seed)
    np.random.seed(seed)

    model = PINN()
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
    errors = {}

    for epoch in range(1, max_epochs + 1):
        x = tf.random.uniform((100, 1), domain_x[0], domain_x[1])
        t = tf.random.uniform((100, 1), domain_t[0], domain_t[1])
        with tf.GradientTape() as tape:
            loss = tf.reduce_mean(tf.square(pde_residual(model, x, t)))
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

        if epoch in checkpoints:
            err = relative_l2_error(model, lambda x, t: np.exp(-t) * np.sin(x))
            errors[epoch] = err
            print(f"  Эпоха {epoch:>5} → L2 = {err:.4e}")

    all_errors[seed] = errors


===== Прогон 1 / 5 =====
  Эпоха  1000 → L2 = 2.7532e-03
  Эпоха  2000 → L2 = 1.5548e-03
  Эпоха  3000 → L2 = 1.1359e-03
  Эпоха  4000 → L2 = 9.2312e-04
  Эпоха  5000 → L2 = 4.4153e-03
  Эпоха  6000 → L2 = 2.1289e-03
  Эпоха  7000 → L2 = 5.6659e-04
  Эпоха  8000 → L2 = 1.5573e-03
  Эпоха  9000 → L2 = 7.9561e-04
  Эпоха 10000 → L2 = 1.8074e-03

===== Прогон 2 / 5 =====
  Эпоха  1000 → L2 = 3.5384e-03
  Эпоха  2000 → L2 = 3.3128e-03
  Эпоха  3000 → L2 = 2.4460e-03
  Эпоха  4000 → L2 = 2.3053e-03
  Эпоха  5000 → L2 = 2.3822e-03
  Эпоха  6000 → L2 = 3.6320e-03
  Эпоха  7000 → L2 = 4.1980e-03
  Эпоха  8000 → L2 = 1.0853e-03
  Эпоха  9000 → L2 = 1.5213e-03
  Эпоха 10000 → L2 = 1.6347e-03

===== Прогон 3 / 5 =====
  Эпоха  1000 → L2 = 4.6046e-03
  Эпоха  2000 → L2 = 1.7243e-03
  Эпоха  3000 → L2 = 6.3318e-03
  Эпоха  4000 → L2 = 2.4209e-03
  Эпоха  5000 → L2 = 2.6087e-03
  Эпоха  6000 → L2 = 7.8020e-04
  Эпоха  7000 → L2 = 5.0919e-03
  Эпоха  8000 → L2 = 2.3759e-03
  Эпоха  9000 → L2 = 1.776

In [14]:
import numpy as np

checkpoints = sorted(next(iter(all_errors.values())).keys())

print(f"{'Эпоха':>6} | {'Среднее L2':>12} | {'Std':>10} | {'Min':>10} | {'Max':>10}")
print("-" * 60)

for epoch in checkpoints:
    vals = [all_errors[s][epoch] for s in all_errors]
    print(f"{epoch:>6} | {np.mean(vals):>12.4e} | {np.std(vals):>10.3e} | "
          f"{min(vals):>10.3e} | {max(vals):>10.3e}")

 Эпоха |   Среднее L2 |        Std |        Min |        Max
------------------------------------------------------------
  1000 |   3.6463e-03 |  5.923e-04 |  2.753e-03 |  4.605e-03
  2000 |   2.2458e-03 |  6.178e-04 |  1.555e-03 |  3.313e-03
  3000 |   3.8666e-03 |  1.883e-03 |  1.136e-03 |  6.332e-03
  4000 |   2.3730e-03 |  9.806e-04 |  9.231e-04 |  4.008e-03
  5000 |   2.6181e-03 |  9.547e-04 |  1.658e-03 |  4.415e-03
  6000 |   1.9570e-03 |  9.865e-04 |  7.802e-04 |  3.632e-03
  7000 |   4.0678e-03 |  1.793e-03 |  5.666e-04 |  5.245e-03
  8000 |   2.1199e-03 |  8.088e-04 |  1.085e-03 |  3.468e-03
  9000 |   1.3452e-03 |  3.367e-04 |  7.956e-04 |  1.776e-03
 10000 |   1.5511e-03 |  4.736e-04 |  6.244e-04 |  1.939e-03


In [16]:
import numpy as np
import plotly.graph_objects as go

checkpoints = sorted(next(iter(all_errors.values())).keys())
means = [np.mean([all_errors[s][e] for s in all_errors]) for e in checkpoints]
stds  = [np.std ([all_errors[s][e] for s in all_errors]) for e in checkpoints]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=checkpoints + checkpoints[::-1],
    y=[m+s for m,s in zip(means,stds)] + [m-s for m,s in zip(means,stds)][::-1],
    fill='toself', fillcolor='rgba(0,100,200,0.2)',
    line=dict(color='rgba(255,255,255,0)'), name='±std'))
fig.add_trace(go.Scatter(
    x=checkpoints, y=means, mode='lines+markers', name='Среднее по 5 seed',
    text=[f'{v:.2e}' for v in means], textposition='top center'))

fig.update_layout(
    title='Влияние числа эпох на L2-ошибку (5 seed)',
    xaxis_title='Число эпох', yaxis_title='Относительная L2-ошибка',
    template='plotly_white')
fig.show()

## Что будет если менять число точек коллокации?

In [20]:
def run_pinn_experiment(n_collocation, seed, epochs=2000):
    tf.random.set_seed(seed)
    np.random.seed(seed)

    model = PINN()
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)

    x_col = tf.random.uniform((n_collocation, 1), domain_x[0], domain_x[1])
    t_col = tf.random.uniform((n_collocation, 1), domain_t[0], domain_t[1])

    for epoch in range(epochs):
        with tf.GradientTape() as tape:
            residual = pde_residual(model, x_col, t_col)
            loss = tf.reduce_mean(tf.square(residual))
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

    return relative_l2_error(model, lambda x, t: np.exp(-t) * np.sin(x))

# Сбор статистики
results = {}
n_values = [50, 100, 200, 500, 1000]
seeds = [1, 2, 3]

for n in n_values:
    errors = [run_pinn_experiment(n, s) for s in seeds]
    results[n] = {"errors": errors, "mean": np.mean(errors), "std": np.std(errors)}
    print(f"N_f={n:>4}: errors={[f'{e:.3e}' for e in errors]}  "
      f"mean={np.mean(errors):.4e} ± {np.std(errors):.2e}")

N_f=  50: errors=['3.776e-03', '7.767e-03', '3.587e-03']  mean=5.0436e-03 ± 1.93e-03
N_f= 100: errors=['8.116e-03', '3.134e-03', '1.450e-03']  mean=4.2335e-03 ± 2.83e-03
N_f= 200: errors=['2.273e-03', '1.625e-03', '1.513e-03']  mean=1.8036e-03 ± 3.35e-04
N_f= 500: errors=['2.223e-03', '3.024e-03', '4.741e-03']  mean=3.3292e-03 ± 1.05e-03
N_f=1000: errors=['1.217e-03', '1.445e-03', '1.557e-03']  mean=1.4066e-03 ± 1.42e-04


In [21]:
for n, r in results.items():
    print(f"N_f={n}: {r['errors']} → mean={r['mean']:.4e}, std={r['std']:.2e}")

N_f=50: [np.float64(0.0037762811750400834), np.float64(0.007767421409753718), np.float64(0.0035870511881676125)] → mean=5.0436e-03, std=1.93e-03
N_f=100: [np.float64(0.008115966293767513), np.float64(0.0031343506674367445), np.float64(0.0014501309537063407)] → mean=4.2335e-03, std=2.83e-03
N_f=200: [np.float64(0.0022731750430984715), np.float64(0.0016248997477130076), np.float64(0.0015126377823526842)] → mean=1.8036e-03, std=3.35e-04
N_f=500: [np.float64(0.002223174552923434), np.float64(0.0030235823883496397), np.float64(0.004740885566453206)] → mean=3.3292e-03, std=1.05e-03
N_f=1000: [np.float64(0.0012168685496373077), np.float64(0.0014454085597052893), np.float64(0.0015574899419683365)] → mean=1.4066e-03, std=1.42e-04


## Что будет если менять оптимизаторы?

In [25]:
import time

def run_with_optimizer(optimizer_name, seed, n_collocation=200, epochs=2000, learning_rate=1e-3):
    tf.random.set_seed(seed)
    np.random.seed(seed)

    model = PINN()

    if optimizer_name == "Adam":
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == "SGD":
        optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)
    elif optimizer_name == "Adagrad":
        optimizer = tf.keras.optimizers.Adagrad(learning_rate=learning_rate)
    else:
        raise ValueError(f"Unknown optimizer: {optimizer_name}")

    x_col = tf.random.uniform((n_collocation, 1), domain_x[0], domain_x[1])
    t_col = tf.random.uniform((n_collocation, 1), domain_t[0], domain_t[1])

    for epoch in range(epochs):
        with tf.GradientTape() as tape:
            residual = pde_residual(model, x_col, t_col)
            loss = tf.reduce_mean(tf.square(residual))
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

    return relative_l2_error(model, lambda x, t: np.exp(-t) * np.sin(x))


optimizers_to_test = ["Adam", "SGD", "Adagrad"]
seeds = [1, 2, 3]

results_opt = {}

for opt_name in optimizers_to_test:
    t0 = time.time()
    errors = [run_with_optimizer(opt_name, s, 200, 2000, 1e-3) for s in seeds]
    results_opt[opt_name] = {
        "errors": errors,
        "mean": np.mean(errors),
        "std": np.std(errors),
    }
    print(f"{opt_name:>8}: errors={[f'{e:.3e}' for e in errors]}  "
          f"mean={np.mean(errors):.4e} ± {np.std(errors):.2e}  "
          f"({time.time()-t0:.1f} s)")

    Adam: errors=['1.522e-03', '1.229e-03', '1.729e-03']  mean=1.4937e-03 ± 2.05e-04  (457.6 s)
     SGD: errors=['1.360e-02', '1.419e-02', '1.050e-02']  mean=1.2763e-02 ± 1.62e-03  (300.2 s)
 Adagrad: errors=['4.525e-02', '9.014e-03', '4.068e-02']  mean=3.1648e-02 ± 1.61e-02  (338.7 s)


In [26]:
for opt_name, r in results_opt.items():
    print(f"{opt_name:>8}: mean={r['mean']:.4e}, std={r['std']:.2e}")

    Adam: mean=1.4937e-03, std=2.05e-04
     SGD: mean=1.2763e-02, std=1.62e-03
 Adagrad: mean=3.1648e-02, std=1.61e-02
